In [2]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import glob

In [ ]:
def stereoCalibration(left_images_directory, right_images_directory, checkerboards_dims: list, visualize=True):
    all_left_images = []
    all_right_images = []
    all_left_points = []
    all_right_points = []
    j = len(checkerboards_dims)
    i = 1
    for checkerboard_dims in checkerboards_dims:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

        objp = np.zeros((checkerboard_dims[0] * checkerboard_dims[1], 3), np.float32)
        objp[:, :2] = np.mgrid[0:checkerboard_dims[0], 0:checkerboard_dims[1]].T.reshape(-1, 2)

        left_images_paths = sorted(glob.glob(left_images_directory + r"/*.png"))
        right_images_paths = sorted(glob.glob(right_images_directory + r"/*.png"))

        if len(left_images_paths) != len(right_images_paths):
            raise Exception("Mismatch in the number of left and right images")

        imgpoints_left = [] 
        imgpoints_right = [] 
        objpoints_left = [] 
        objpoints_right = [] 

        for left_img_file, right_img_path in zip(left_images_paths, right_images_paths):
            
            left_img = cv2.imread(left_img_file)
            right_img = cv2.imread(right_img_path)
            gray_left = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
            gray_right = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)

            ret_left, corners_left = cv2.findChessboardCorners(gray_left, checkerboard_dims, None)
            ret_right, corners_right = cv2.findChessboardCorners(gray_right, checkerboard_dims, None)

            if ret_left and ret_right:
                # corners_left = cv2.cornerSubPix(gray_left, corners_left, (11, 11), (-1, -1), criteria)
                # corners_right = cv2.cornerSubPix(gray_right, corners_right, (11, 11), (-1, -1), criteria)

                all_left_images.append(left_img)
                all_right_images.append(right_img)
                imgpoints_left.append(corners_left)
                imgpoints_right.append((right_img, corners_right))
        all_left_points.append(imgpoints_left)
        all_right_points.append(imgpoints_right)
        print(f"{int(i/j*100)}" + "%")
        i += 1
    if visualize:
        last_left_img = left_img
        last_right_img = right_img
        last_left_corners = corners_left
        last_right_corners = corners_right
                # objpoints = objp
        # Draw the 3D object points on the last image pair
        cv2.drawChessboardCorners(last_left_img, checkerboards_dims[-1], last_left_corners, True)
        cv2.drawChessboardCorners(last_right_img, checkerboards_dims[-1], last_right_corners, True)

        combined_image = np.hstack((last_left_img, last_right_img))
        cv2.imshow("Last Calibration Image Pair with Object Points", combined_image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    cv2.destroyAllWindows()

left = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_02\data"
right = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_03\data"

corners = [
    (11, 7), (11, 7), (7, 5), (5, 7),
    (7, 5), (5, 7), (5, 7), (7, 5),
    (7, 5), (7, 5), (5, 7), (5, 7), (5, 15)
]

stereoCalibration(left, right, corners, visualize=True)


7%
15%
23%
30%
38%
46%
53%
61%
69%
76%
84%
92%
100%


In [4]:
import cv2
import glob
import numpy as np

def processStereoImages(left_image_path, right_image_path, checkerboards_dims: list, visualize=True):
    left_img = cv2.imread(left_image_path)
    right_img = cv2.imread(right_image_path)
    gray_left = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)
    
    all_corners_left = []
    all_corners_right = []
    
    for checkerboard_dims in checkerboards_dims:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        
        objp = np.zeros((checkerboard_dims[0] * checkerboard_dims[1], 3), np.float32)
        objp[:, :2] = np.mgrid[0:checkerboard_dims[0], 0:checkerboard_dims[1]].T.reshape(-1, 2)
        
        ret_left, corners_left = cv2.findChessboardCorners(gray_left, checkerboard_dims, None)
        ret_right, corners_right = cv2.findChessboardCorners(gray_right, checkerboard_dims, None)
        
        if ret_left and ret_right:
            # corners_left = cv2.cornerSubPix(gray_left, corners_left, (11, 11), (-1, -1), criteria)
            # corners_right = cv2.cornerSubPix(gray_right, corners_right, (11, 11), (-1, -1), criteria)
            
            all_corners_left.append(corners_left)
            all_corners_right.append(corners_right)
    
    if visualize:
        for corners in all_corners_left:
            cv2.drawChessboardCorners(left_img, checkerboards_dims[0], corners, True)
        cv2.imshow("Left Image with All Corners", left_img)
        cv2.waitKey(1000)
        cv2.destroyAllWindows()
    
    return left_img, right_img,all_corners_left,all_corners_right

def stereoCalibration(left_img_dir,right_img_dir,corners,visualize=True):
    left_image_files = glob.glob(left_img_dir+r"\*.png")
    right_image_files = glob.glob(right_img_dir+r"\*.png")
    # print(left_image_files)
    left_images = []
    right_images = []
    corners_left = []
    corners_right = []
    for left_image_file, right_image_file in zip(left_image_files,right_image_files):
        l,r,cl,cr =processStereoImages(left_image_file,right_image_file,corners,visualize = False)
        left_images.append(l)
        right_images.append(r)
        corners_left.append(cl)
        corners_right.append(cr)
    
    if visualize:
        left_image = left_images[-1]
        cs_l = corners_left[-1]
        right_image = right_images[-1]
        cs_r = corners_right[-1]
        for c_l,c_r,c in zip(cs_l,cs_r,corners):
            cv2.drawChessboardCorners(left_image, c, c_l, True)
            cv2.drawChessboardCorners(right_image, c, c_r, True)
            print(c)

        cv2.imshow("Left Image with All Corners", left_image)
        cv2.imshow("Right Image with All Corners",right_image)
        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    cv2.destroyAllWindows()


# Example usage:
left_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_02\data"
right_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_03\data"

corners = [
    (11, 7), (11, 7), (7, 5), (5, 7),
    (7, 5), (5, 7), (5, 7), (7, 5),
    (7, 5), (7, 5), (5, 7), (5, 7), (5, 15)
]

stereoCalibration(left_img_path, right_img_path, corners, visualize=True)


(11, 7)
(11, 7)
(7, 5)
(5, 7)
(7, 5)
(5, 7)
(5, 7)
(7, 5)
(7, 5)
(7, 5)
(5, 7)
(5, 7)


In [ ]:
import cv2
import glob
import numpy as np

def processStereoImages(left_image_path, right_image_path, checkerboards_dims: list, visualize=True):
    left_img = cv2.imread(left_image_path)
    right_img = cv2.imread(right_image_path)
    gray_left = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)
    
    all_corners_left = []
    all_corners_right = []
    
    for checkerboard_dims in checkerboards_dims:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

        objp = np.zeros((checkerboard_dims[0] * checkerboard_dims[1], 3), np.float32)
        objp[:, :2] = np.mgrid[0:checkerboard_dims[0], 0:checkerboard_dims[1]].T.reshape(-1, 2)

        # Clone images to apply masking
        mask_left = gray_left.copy()
        mask_right = gray_right.copy()
        
        while True:
            # Find checkerboard corners in the masked regions
            ret_left, corners_left = cv2.findChessboardCorners(mask_left, checkerboard_dims, None)
            ret_right, corners_right = cv2.findChessboardCorners(mask_right, checkerboard_dims, None)

            if ret_left and ret_right:
                # Append detected corners
                all_corners_left.append(corners_left)
                all_corners_right.append(corners_right)

                # Mask out detected regions to prevent repeated detections
                mask_left = cv2.fillConvexPoly(mask_left, corners_left.astype(int), 0)
                mask_right = cv2.fillConvexPoly(mask_right, corners_right.astype(int), 0)
            else:
                break

    if visualize:
        for corners in all_corners_left:
            cv2.drawChessboardCorners(left_img, checkerboards_dims[0], corners, True)
        cv2.imshow("Left Image with All Corners", left_img)
        cv2.waitKey(1000)
        cv2.destroyAllWindows()

    return left_img, right_img, all_corners_left, all_corners_right

def stereoCalibration(left_img_dir, right_img_dir, corners, visualize=True):
    left_image_files = glob.glob(left_img_dir + r"\*.png")
    right_image_files = glob.glob(right_img_dir + r"\*.png")

    left_image_files = [left_image_files[-1]]
    right_image_files = [right_image_files[-1]]


    left_images = []
    right_images = []
    corners_left = []
    corners_right = []

    for left_image_file, right_image_file in zip(left_image_files, right_image_files):
        l, r, cl, cr = processStereoImages(left_image_file, right_image_file, corners, visualize=False)
        left_images.append(l)
        right_images.append(r)
        corners_left.append(cl)
        corners_right.append(cr)

    if visualize:
        left_image = left_images[-1]
        cs_l = corners_left[-1]
        right_image = right_images[-1]
        cs_r = corners_right[-1]
        for c_l, c_r, c in zip(cs_l, cs_r, corners):
            cv2.drawChessboardCorners(left_image, c, c_l, True)
            cv2.drawChessboardCorners(right_image, c, c_r, True)

        cv2.imshow("Final Left Image with All Corners", left_image)
        cv2.imshow("Final Right Image with All Corners", right_image)

        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cv2.destroyAllWindows()

# Example usage:
left_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_02\data"
right_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_03\data"

corners = [
    (11, 7), (11, 7), 
    (7, 5), (7, 5), (7, 5), (7, 5), (7, 5), 
    (5, 7), (5, 7), (5, 7), (5, 7), (5, 7),
    (5, 15)
]

stereoCalibration(left_img_path, right_img_path, corners, visualize=True)


In [21]:
import cv2
import glob
import numpy as np

def find_all_checkerboards_with_masks_enhanced(image, checkerboards_dims):
    detected_corners = []
    mask = np.ones_like(image, dtype=np.uint8)  # Initialize mask

    for checkerboard_dims in checkerboards_dims:
        while True:
            masked_image = cv2.bitwise_and(image, image, mask=mask)

            # Pre-process the image to improve contrast
            enhanced_image = cv2.equalizeHist(masked_image)

            # Find checkerboards with improved flags
            ret, corners = cv2.findChessboardCorners(enhanced_image, checkerboard_dims, 
                                                     flags=cv2.CALIB_CB_ADAPTIVE_THRESH + 
                                                           cv2.CALIB_CB_NORMALIZE_IMAGE)

            if ret:
                # Refine corners for accuracy
                # criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
                # corners = cv2.cornerSubPix(enhanced_image, corners, (11, 11), (-1, -1), criteria)
                
                detected_corners.append((corners, checkerboard_dims))

                # Mask out the detected checkerboard region (use convex hull for better masking)
                polygon_points = np.array([corner[0] for corner in corners], dtype=np.int32)
                cv2.fillConvexPoly(mask, polygon_points, 0)
            else:
                break

    return detected_corners

def processStereoImages(left_image_path, right_image_path, checkerboards_dims: list, visualize=True):
    left_img = cv2.imread(left_image_path)
    right_img = cv2.imread(right_image_path)
    gray_left = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)

    all_corners_left = find_all_checkerboards_with_masks_enhanced(gray_left, checkerboards_dims)
    all_corners_right = find_all_checkerboards_with_masks_enhanced(gray_right, checkerboards_dims)

    if visualize:
        for corners, dims in all_corners_left:
            cv2.drawChessboardCorners(left_img, dims, corners, True)
        for corners, dims in all_corners_right:
            cv2.drawChessboardCorners(right_img, dims, corners, True)

        cv2.imshow("Left Image with All Corners", left_img)
        cv2.imshow("Right Image with All Corners", right_img)
        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cv2.destroyAllWindows()

    return left_img, right_img, all_corners_left, all_corners_right

def stereoCalibration(left_img_dir, right_img_dir, corners, visualize=True):
    left_image_files = glob.glob(left_img_dir + r"\*.png")
    right_image_files = glob.glob(right_img_dir + r"\*.png")

    # left_image_files = [left_image_files[-1]]
    # right_image_files = [right_image_files[-1]]

    left_images = []
    right_images = []
    corners_left = []
    corners_right = []

    for left_image_file, right_image_file in zip(left_image_files, right_image_files):
        l, r, cl, cr = processStereoImages(left_image_file, right_image_file, corners, visualize=True)
        left_images.append(l)
        right_images.append(r)
        corners_left.append(cl)
        corners_right.append(cr)

    if visualize:
        left_image = left_images[-1]
        cs_l = corners_left[-1]
        right_image = right_images[-1]
        cs_r = corners_right[-1]
        for c_l, dims_l in cs_l:
            cv2.drawChessboardCorners(left_image, dims_l, c_l, True)
        for c_r, dims_r in cs_r:
            cv2.drawChessboardCorners(right_image, dims_r, c_r, True)

        cv2.imshow("Final Left Image with All Corners", left_image)
        cv2.imshow("Final Right Image with All Corners", right_image)

        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cv2.destroyAllWindows()

# Example usage:
left_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_02\data"
right_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_03\data"

corners = [
    (7,11), (7,11), 
    (5,7), (5,7), (5,7), (5,7), (5,7), 
    (7,5), (7,5), (7,5), (7,5), (7,5), 
    (15, 5)
]
# corners = [
#     (11, 7), (11, 7),
#     (7, 5), (7, 5), (7, 5), (7, 5), (7, 5),
#     (5, 7), (5, 7), (5, 7), (5, 7), (5, 7), 
#     (5, 15)
# ]
stereoCalibration(left_img_path, right_img_path, corners, visualize=True)


KeyboardInterrupt: 